In [6]:
import requests
import json
from uuid import UUID
import uuid


BASE_URL = "http://localhost:8000"
user_id = "d866155a-9e57-423e-b102-5d74fe24fe22"
thread_id = "35cd6e32-c370-48a7-80dd-72d0cc3486d4"

In [7]:
message_data = {
    "sender": "human",
    "content": "Hello, this is my first message!",
    "is_image": False,
    "metadata": {"mood": "happy"},
}

response = requests.post(f"{BASE_URL}/threads/{thread_id}/messages", json=message_data)
print("Status:", response.status_code)
print("Response:", response.json())

message = response.json()

Status: 200
Response: {'id': '1e38af42-7c4e-4634-9fa0-b4d77f9a6a61', 'thread_id': '35cd6e32-c370-48a7-80dd-72d0cc3486d4', 'sender': 'assistant', 'content': 'Hi! Great to meet you. I can help you explore and analyze your data conversationally, run SQL on demand, and create charts.\n\nWe have tables like calls, deals, notes, contacts, and companies. What would you like to look at first?\n- Quick health checks (e.g., recent close rate trend, call connection rates by hour)\n- Pipeline insights (win rates by stage, owner performance)\n- Marketing/source performance (which sources convert best)\n- Data quality checks (duplicates, field inconsistencies)\n\nIf you prefer, I can start with a brief overview of recent activity and trends, or build a specific chart. What’s your goal today?', 'is_image': False, 'base64_image': '', 'extra_metadata': {'from_agent': True}, 'created_at': '2025-10-26T22:31:31.829514Z'}


In [ ]:
user_data = {"email": "a_square@gmail.com", "display_name": "A_Square010"}

response = requests.post(f"{BASE_URL}/users", json=user_data)
print("Status:", response.status_code)
print("Response:", response.json())

user = response.json()
user_id = user["id"]

In [ ]:
thread_data = {
    "title": f"Thread {uuid.uuid4()}",
    "extra_metadata": {"topic": "general"},
}

response = requests.post(f"{BASE_URL}/users/{user_id}/threads", json=thread_data)
print("Status:", response.status_code)
# print("Response:", response.json())

thread = response.json()
thread_id = thread["id"]

Status: 200


In [19]:
response = requests.get(f"{BASE_URL}/threads/{thread_id}")
print("Status:", response.status_code)
print("Response:", json.dumps(response.json(), indent=2))

Status: 200
Response: {
  "id": "35cd6e32-c370-48a7-80dd-72d0cc3486d4",
  "user_id": "d866155a-9e57-423e-b102-5d74fe24fe22",
  "title": "Thread 4c104987-c11b-4737-99b6-101a45fc7890",
  "extra_metadata": {
    "topic": "general"
  },
  "created_at": "2025-10-25T23:22:14.427467Z",
  "updated_at": "2025-10-25T23:22:14.427467Z",
  "messages": [
    {
      "id": "1746e28a-fa45-421a-9218-db1f796471a6",
      "thread_id": "35cd6e32-c370-48a7-80dd-72d0cc3486d4",
      "sender": "human",
      "content": "Hello, this is my first message!",
      "is_image": false,
      "base64_image": null,
      "extra_metadata": {},
      "created_at": "2025-10-25T23:25:51.844642Z"
    }
  ]
}


In [20]:
fake_thread_id = str(uuid.uuid4())
response = requests.get(f"{BASE_URL}/threads/{fake_thread_id}")
print("Status:", response.status_code)
print("Response:", response.json())

Status: 404
Response: {'detail': 'Thread not found'}


In [ ]:
# imports (add to top of your file)
from uuid import UUID
from fastapi import FastAPI, Depends, HTTPException
from starlette.concurrency import run_in_threadpool
from sqlalchemy.ext.asyncio import AsyncSession
from backend_db import crud, get_db
from common.models.api_models import MessageRead, MessageCreate
from backend_db.models import Sender
from agentic.chat_bot import ChatOrchestrator
from common.utils.postgres_client import PostgresClient
from common.models.db_metadata import SchemaMetadata
import json
from pathlib import Path

app = FastAPI(title="Chat Service")


# ---------- INIT: create a global orchestrator at startup ----------
# adapt this to load your real SchemaMetadata and DB client
@app.on_event("startup")
async def startup_event():
    # example: load metadata from file / build PostgresClient
    repo_root = Path(__file__).resolve().parents[3]
    example_metadata_path = repo_root / "db_faker" / "data" / "metadata_dump.json"
    if not example_metadata_path.exists():
        # if you don't have metadata JSON, create metadata another way
        raise FileNotFoundError(
            "metadata_dump.json not found; initialize your metadata properly"
        )

    with example_metadata_path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    metadata = SchemaMetadata.model_validate(data)
    db_client = PostgresClient()
    db_client.connect()

    # store orchestrator on app.state so handlers can access it
    app.state.orchestrator = ChatOrchestrator(db_client=db_client, metadata=metadata)


# helper dependency to access orchestrator
def get_orchestrator():
    return app.state.orchestrator


# ---------- Endpoint: post_message that returns the AI reply ----------
@app.post("/threads/{thread_id}/messages", response_model=MessageRead)
async def post_message(
    thread_id: UUID,
    message_in: MessageCreate,
    db: AsyncSession = Depends(get_db),
    orchestrator: ChatOrchestrator = Depends(get_orchestrator),
):
    # ensure thread exists
    thread = await crud.get_thread_with_messages(db, thread_id)
    if not thread:
        raise HTTPException(status_code=404, detail="Thread not found")

    # 1) create human message record first
    human_msg = await crud.create_message(
        db,
        thread_id=thread_id,
        sender=message_in.sender,  # probably Sender.human
        content=message_in.content,
        is_image=message_in.is_image,
        base64_image=message_in.base64_image,
        extra_metadata=message_in.extra_metadata,
    )

    # 2) call the ChatOrchestrator (blocking) in a threadpool to avoid blocking event loop
    #    orchestrator.invoke returns the assistant's textual reply; the agent may set base64_image on itself
    try:
        ai_text = await run_in_threadpool(orchestrator.invoke, message_in.content or "")
    except Exception as e:
        # optionally create an error assistant message or return 500
        raise HTTPException(status_code=500, detail=f"AI generation failed: {e}")

    # 3) create assistant message using the AI reply
    assistant_msg = await crud.create_message(
        db,
        thread_id=thread_id,
        sender=Sender.assistant,  # create as assistant
        content=ai_text,
        is_image=bool(getattr(orchestrator, "base64_image", "")),
        base64_image=getattr(orchestrator, "base64_image", None),
        extra_metadata={"from_agent": True},  # add whatever metadata you want
    )

    # 4) Convert to Pydantic model while session is active (avoid lazy I/O later)
    #    Use model_validate for pydantic v2 (your models should have model_config={"from_attributes": True})
    msg_out = MessageRead.model_validate(assistant_msg)

    return msg_out